# CellIQ tissue model v3 (retrain for the app)

Trains the lung-slide model with the **v2 grayscale-robust recipe** (stronger grayscale augmentation, label smoothing, cosine schedule, 4 + 8 epochs) on our **near-duplicate-aware split**, then writes `model.pt`, `meta.json` and `metrics.json` in exactly the format the app loads.

Changes from the model currently in the app:
- Recipe: the v2 grayscale-robust recipe (it won on all three test groups we repeated).
- Cancer-vs-normal cut-off: the middle of the cut-offs that separate normal from cancer best on the validation images. (The old "highest cut-off that keeps 95% sensitivity" rule sits at the edge once label smoothing squeezes the scores; the 95% and 99% versions are still reported on the test images for comparison.)
- Same split as before (test group 0, validation group 1), so the numbers compare directly with the app model.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 25 minutes.

Research and educational prototype. Not for real diagnosis.

## 1. Settings

In [ ]:
DATASET = "lung_slides"
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
HEAD_EPOCHS = 4
FINETUNE_EPOCHS = 8
SENS_TARGET = 0.99              # share of cancer slides the cut-off must catch on the validation images
OUT_DIR = "/content/cellIQ_lung_slides_v3"

## 2. Tools, data and file list (same code as our tissue notebook)

In [ ]:
import os, json, random, re, shutil
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
os.makedirs(OUT_DIR, exist_ok=True)
from concurrent.futures import ThreadPoolExecutor

In [ ]:
import kagglehub
path = kagglehub.dataset_download("javaidahmadwani/lc25000")
# The download holds several copies of the lung folders (for example a small "Test Set" with 1,499 images and the full set).
# Pick the folder with the most images, never just the first match.
def count_images(folder):
    return sum(len([f for f in os.listdir(os.path.join(folder, d)) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
               for d in ("lung_aca", "lung_n", "lung_scc"))
cands = [root for root, dirs, _ in os.walk(path) if {"lung_aca", "lung_n", "lung_scc"} <= set(dirs)]
for c in cands:
    print(count_images(c), "images in", c)
assert cands, "lung folders not found"
DATA_DIR = max(cands, key=count_images)
print("DATA_DIR =", DATA_DIR, "|", count_images(DATA_DIR), "images")
assert count_images(DATA_DIR) >= 10000, "expected the full 15,000-image lung set"

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
class_names = sorted([d for d in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, d)) and d.startswith("lung_")])
print("Groups:", class_names)
natural = lambda s: [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]
files, labels = [], []
for idx, d in enumerate(class_names):
    folder = os.path.join(DATA_DIR, d)
    for f in sorted(os.listdir(folder), key=natural):
        if f.lower().endswith(IMG_EXT):
            files.append(os.path.join(folder, f)); labels.append(idx)
files = np.array(files); labels = np.array(labels)
print("Total images:", len(files), "| per group:", np.bincount(labels))

## 3. Load every image once at 224 x 224

In [ ]:
def load_resized(path):
    return np.asarray(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))

with ThreadPoolExecutor(max_workers=8) as ex:
    CACHE = np.stack(list(ex.map(load_resized, files)))
print("cached images:", CACHE.shape)

## 4. Group near-duplicates and split by group (our method, unchanged)

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
emb_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])

class PlainDS(Dataset):
    def __len__(self): return len(files)
    def __getitem__(self, i): return emb_tf(Image.fromarray(CACHE[i])), 0

emb = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).to(device).eval()
emb.classifier = nn.Identity()
feats = []
with torch.no_grad():
    for x, _ in DataLoader(PlainDS(), batch_size=64, num_workers=2):
        feats.append(emb(x.to(device)).float().cpu())
feats = torch.cat(feats)
feats = F.normalize(feats - feats.mean(0), dim=1).to(device)
N = len(feats)

FLOOR = 0.75
pi, pj, ps = [], [], []
for i in range(0, N, 1024):
    s = feats[i:i+1024] @ feats.T
    r, c = torch.where(s >= FLOOR)
    pi.append((r + i).cpu().numpy()); pj.append(c.cpu().numpy()); ps.append(s[r, c].cpu().numpy())
pi, pj, ps = np.concatenate(pi), np.concatenate(pj), np.concatenate(ps)
print("Linked pairs at similarity >=", FLOOR, ":", len(pi))

def components(thr):
    m = ps >= thr
    G = coo_matrix((np.ones(m.sum()), (pi[m], pj[m])), shape=(N, N))
    n, gid = connected_components(G, directed=False)
    return n, gid

table = {}
for thr in (0.80, 0.825, 0.85, 0.875, 0.90, 0.925, 0.95):
    n, gid = components(thr)
    sizes = np.bincount(gid)
    table[thr] = (n, gid, int(sizes.max()))
    print(f"similarity >= {thr:.3f}: groups={n:5d} | biggest group={sizes.max():5d} | median size={int(np.median(sizes))}")

# Linking images by similarity can chain unrelated photos into one giant group (which breaks the split).
# Use the LOWEST threshold (most careful grouping) whose biggest group is still small enough to split fairly.
MAX_GROUP = 700
ok_thr = [t for t in table if table[t][2] <= MAX_GROUP]
SIM = min(ok_thr) if ok_thr else max(table)
n_groups, group_id, _ = table[SIM]
print("Chosen similarity threshold:", SIM, "->", n_groups, "groups")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = [idx for _, idx in sgkf.split(files, labels, group_id)]
te, va = folds[0], folds[1]
tr = np.concatenate(folds[2:])
for nm, ix in (("train", tr), ("val", va), ("test", te)):
    print(f"{nm:5s} n={len(ix):5d} per group: {np.bincount(labels[ix], minlength=len(class_names))}")

# leakage check: how many test images still have a similar-looking twin in train? (random split vs group split)
def twin_rates(test_idx, train_idx, thrs=(0.95, 0.90, 0.85, 0.80)):
    best = []
    for i in range(0, len(test_idx), 512):
        s = feats[test_idx[i:i+512]] @ feats[train_idx].T
        best.append(s.max(1).values.cpu().numpy())
    best = np.concatenate(best)
    return {str(t): round(float((best >= t).mean()), 4) for t in thrs}

rng = np.random.RandomState(SEED); perm = rng.permutation(N)
r_te, r_tr = perm[:len(te)], perm[len(te) + len(va):]
leak_random = twin_rates(r_te, r_tr)
leak_group = twin_rates(te, tr)
print("Share of test images with a near-identical twin in train, at similarity >= 0.95 / 0.90 / 0.85 / 0.80:")
print("  random split:", leak_random)
print("  group split :", leak_group, "(zero at or above the grouping threshold by construction; lower levels show what remains)")

## 5. Train with the v2 recipe

In [ ]:
train_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomGrayscale(p=0.5), transforms.RandomHorizontalFlip(),
                               transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.25, contrast=0.25),
                               transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])
gray_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.Grayscale(3), transforms.ToTensor(), transforms.Normalize(mean, std)])
K = len(class_names)


class CachedDS(Dataset):
    def __init__(self, idxs, tf):
        self.idxs, self.tf = idxs, tf
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i):
        j = self.idxs[i]
        return self.tf(Image.fromarray(CACHE[j])), int(labels[j])


torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, K)
model = model.to(device)
train_dl = DataLoader(CachedDS(tr, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_dl = DataLoader(CachedDS(va, eval_tf), batch_size=BATCH_SIZE, num_workers=2)
cnt = np.bincount(labels[tr], minlength=K)
w = torch.tensor(cnt.sum() / (K * cnt), dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
best = {"loss": 1e9, "state": None}


def run_epoch(dl, train, opt=None):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); total += len(y)
            correct += (out.argmax(1) == y).sum().item()
    return loss_sum / total, correct / total


def fit(epochs, lr, name):
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        tl, ta = run_epoch(train_dl, True, opt)
        vl, va_ = run_epoch(val_dl, False)
        sched.step()
        print(f"{name} epoch {ep + 1}: train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va_:.3f}", flush=True)
        if vl < best["loss"]:
            best["loss"] = vl
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


for p in model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True
fit(HEAD_EPOCHS, 1e-3, "head")
for p in model.parameters():
    p.requires_grad = True
fit(FINETUNE_EPOCHS, 1e-4, "fine-tune")
model.load_state_dict(best["state"])
os.makedirs(OUT_DIR, exist_ok=True)
torch.save(best["state"], os.path.join(OUT_DIR, "model.pt"))
print("Training finished and best model saved.")

## 6. Test it

Same checks as the app model: accuracy on the held-out group, accuracy with colour removed, per-class results, and the cancer-vs-normal risk score (risk = 1 - P(normal)). The cut-off is picked on the validation images only; the test images are never used to choose it.

In [ ]:
@torch.no_grad()
def get_probs(idx, tf):
    model.eval()
    P, Y = [], []
    for x, y in DataLoader(CachedDS(idx, tf), batch_size=64, num_workers=2):
        P.append(F.softmax(model(x.to(device)), dim=1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

pv, yv = get_probs(va, eval_tf)
pt, yt = get_probs(te, eval_tf)
pg, _ = get_probs(te, gray_tf)
pred, pred_g = pt.argmax(1), pg.argmax(1)

print(classification_report(yt, pred, target_names=class_names, digits=3))
cm = confusion_matrix(yt, pred)
print("Confusion matrix (rows = truth, columns = AI guess):\n", cm)
acc, acc_gray = float((pred == yt).mean()), float((pred_g == yt).mean())
wrong = int((pred != yt).sum())
print(f"Test accuracy (group split): {acc:.4f} | with colour removed: {acc_gray:.4f} | wrong: {wrong} of {len(yt)}")
print("Multi-class AUC (one-vs-rest):", round(float(roc_auc_score(yt, pt, multi_class="ovr")), 4))

n_idx = class_names.index("lung_n")
risk_v, risk_t = 1 - pv[:, n_idx], 1 - pt[:, n_idx]
yb_v, yb_t = (yv != n_idx).astype(int), (yt != n_idx).astype(int)
grid = np.linspace(0.005, 0.995, 199)

def pick_threshold(target):
    ok = [t for t in grid if (risk_v[yb_v == 1] >= t).mean() >= target]
    return float(max(ok)) if ok else 0.5

def binary_result(thr):
    tn, fp, fn, tp = confusion_matrix(yb_t, (risk_t >= thr).astype(int), labels=[0, 1]).ravel()
    return dict(threshold=round(float(thr), 3), sensitivity=round(float(tp / max(tp + fn, 1)), 4),
                specificity=round(float(tn / max(tn + fp, 1)), 4), missed_cancers=int(fn), false_alarms=int(fp))

auc_bin = round(float(roc_auc_score(yb_t, risk_t)), 4)
rules = {f"cut-off for >= {int(t * 100)}% validation sensitivity": binary_result(pick_threshold(t)) for t in (0.95, SENS_TARGET)}
for name, r in rules.items():
    print(name, "->", r)
print("Cancer-vs-normal AUC:", auc_bin)
# Rule used for the app: the middle of the cut-offs that separate normal from cancer best on the validation images
# (sensitivity + specificity - 1). The "highest cut-off" rules above sit at the edge when label smoothing squeezes the scores.
J = [(risk_v[yb_v == 1] >= t).mean() + (risk_v[yb_v == 0] < t).mean() - 1 for t in grid]
best_t = [float(t) for t, j in zip(grid, J) if j >= max(J) - 1e-9]
print("validation: best cut-offs range from", round(min(best_t), 3), "to", round(max(best_t), 3))
thr = round(float(np.median(best_t)), 3)
chosen = binary_result(thr)
bin_res = dict(chosen, auc=auc_bin, rule="middle of the cut-offs that best separate normal from cancer on the validation images")
print("CHOSEN for the app:", bin_res)

## 7. Heatmap examples and the confusion matrix

In [ ]:
acts = {}
def fwd_hook(module, inputs, output):          # must return nothing, or it would replace the layer output
    acts["a"] = output
    output.register_hook(lambda g: acts.__setitem__("g", g))
hook = model.features[-1].register_forward_hook(fwd_hook)

def gradcam(j):
    x = eval_tf(Image.fromarray(CACHE[j])).unsqueeze(0).to(device)
    model.eval(); model.zero_grad()
    out = model(x); c = out.argmax(1).item()
    out[0, c].backward()
    wgt = acts["g"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((wgt * acts["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
    cam = ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).detach().cpu().numpy()
    return cam, c, F.softmax(out, dim=1)[0].detach().cpu().numpy()

pick = random.Random(SEED).sample(list(te), 6)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for k, j in enumerate(pick):
    cam, c, prob = gradcam(j)
    ax1, ax2 = axes[k // 2][(k % 2) * 2], axes[k // 2][(k % 2) * 2 + 1]
    ax1.imshow(CACHE[j]); ax1.set_title("Truth: " + class_names[labels[j]], fontsize=9); ax1.axis("off")
    ax2.imshow(CACHE[j]); ax2.imshow(cam, cmap="jet", alpha=0.45)
    ax2.set_title(f"AI: {class_names[c]} ({prob[c] * 100:.0f}%)", fontsize=9); ax2.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "heatmap_examples.png"), dpi=130); plt.show()
hook.remove()

fig, ax = plt.subplots(figsize=(4.5, 4)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(K)); ax.set_yticks(range(K))
ax.set_xticklabels(class_names, rotation=30, ha="right"); ax.set_yticklabels(class_names)
for i in range(K):
    for j in range(K):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("AI guess"); ax.set_ylabel("Truth"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150); plt.show()

## 8. Save the files the app loads

In [ ]:
metrics = dict(test_accuracy=round(acc, 4), test_accuracy_grayscale=round(acc_gray, 4), n_test=int(len(te)), wrong=wrong,
               cancer_vs_normal=bin_res, other_cutoff_rules=rules,
               recipe="v2 grayscale-robust: RandomGrayscale 0.5, flip, rot 10, ColorJitter(0.25, 0.25), label smoothing 0.1, cosine LR, 4 head + 8 fine-tune epochs",
               similarity_threshold_for_groups=SIM, n_groups=int(n_groups),
               leak_rate_random_split=leak_random, leak_rate_group_split=leak_group,
               confusion_matrix=cm.tolist(), classes=class_names,
               note="Group-aware split (near-duplicate copies kept together). Still one public dataset: not clinically validated.")
json.dump(metrics, open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)
json.dump({"classes": class_names, "img_size": IMG_SIZE, "dataset": DATASET, "risk": "1 - P(lung_n)", "threshold": thr,
           "normalize_mean": mean, "normalize_std": std, "preprocess": "resize to 224x224, ImageNet normalise"},
          open(os.path.join(OUT_DIR, "meta.json"), "w"), indent=2)
shutil.make_archive("/content/cellIQ_lung_slides_v3", "zip", OUT_DIR)
print("Zip ready:", "/content/cellIQ_lung_slides_v3.zip", round(os.path.getsize("/content/cellIQ_lung_slides_v3.zip") / 1e6, 1), "MB")
print("\nSUMMARY")
print(json.dumps({k: v for k, v in metrics.items() if k not in ("confusion_matrix", "leak_rate_random_split", "leak_rate_group_split")}, indent=2))